<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **SpaceX  Falcon 9 first stage Landing Prediction**


# Lab 1: Collecting the data


Estimated time needed: **45** minutes


In this capstone, we will predict if the Falcon 9 first stage will land successfully. SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars; other providers cost upward of 165 million dollars each, much of the savings is because SpaceX can reuse the first stage. Therefore if we can determine if the first stage will land, we can determine the cost of a launch. This information can be used if an alternate company wants to bid against SpaceX for a rocket launch. In this lab, you will collect and make sure the data is in the correct format from an API. The following is an example of a successful and launch.


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/crash.gif)


Most unsuccessful landings are planned. Space X performs a controlled landing in the oceans. 


## Objectives


In this lab, you will make a get request to the SpaceX API. You will also do some basic data wrangling and formating. 

- Request to the SpaceX API
- Clean the requested data


----


## Import Libraries and Define Auxiliary Functions


We will import the following libraries into the lab


In [1]:
# Requests allows us to make HTTP requests which we will use to get data from an API
import requests
# Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
# NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays
import numpy as np
# Datetime is a library that allows us to represent dates
import datetime

# Setting this option will print all collumns of a dataframe
pd.set_option('display.max_columns', None)
# Setting this option will print all of the data in a feature
pd.set_option('display.max_colwidth', None)

Below we will define a series of helper functions that will help us use the API to extract information using identification numbers in the launch data.

From the <code>rocket</code> column we would like to learn the booster name.


In [2]:
# Takes the dataset and uses the rocket column to call the API and append the data to the list
def getBoosterVersion(data):
    for x in data['rocket']:
       if x:
        response = requests.get("https://api.spacexdata.com/v4/rockets/"+str(x)).json()
        BoosterVersion.append(response['name'])

In [2]:
def getBoosterVersion(data):
    # 1. Download the working main dataset file mirror once
    launch_response = requests.get("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json").json()

    # 2. Extract the rocket details embedded directly inside the launch logs
    rocket_map = {}
    for launch in launch_response:
        rocket_data = launch.get('rocket', None)
        
        if isinstance(rocket_data, dict):
            r_id = rocket_data.get('id') or rocket_data.get('rocket_id')
            r_name = rocket_data.get('name')
            if r_id and r_name:
                rocket_map[r_id] = r_name
                
        elif isinstance(rocket_data, str):
            r_name = launch.get('rocket_name')
            if r_name:
                rocket_map[rocket_data] = r_name

    # 3. Bulletproof fallback to static official SpaceX core IDs if network file map fails
    if not rocket_map:
        rocket_map = {
            '5e9d0d95eda69955f709d1eb': 'Falcon 1',
            '5e9d0d95eda69973a809d1ec': 'Falcon 9',
            '5e9d0d95eda69974db09d1ed': 'Falcon Heavy',
            '5e9d0d95eda69974db09d1ee': 'Starship'
        }

    # 4. Run the lookup loop over data['rocket'] and append directly to global list
    for x in data['rocket']:
        if x:
            if isinstance(x, dict):
                booster_name = x.get('name', None)
            else:
                booster_name = rocket_map.get(x, None)
            BoosterVersion.append(booster_name)
        else:
            BoosterVersion.append(None)

From the <code>launchpad</code> we would like to know the name of the launch site being used, the logitude, and the latitude.


In [3]:
# Takes the dataset and uses the launchpad column to call the API and append the data to the list
def getLaunchSite(data):
    for x in data['launchpad']:
       if x:
         response = requests.get("https://api.spacexdata.com/v4/launchpads/"+str(x)).json()
         Longitude.append(response['longitude'])
         Latitude.append(response['latitude'])
         LaunchSite.append(response['name'])

In [3]:
def getLaunchSite(data):
    # Static lookup map for SpaceX launchpads to bypass the broken API
    launchpad_map = {
        '5e9e4501f509094ba4566f84': {
            'name': 'VAFB SLC 4E', 
            'latitude': 34.632839, 
            'longitude': -120.6107455
        },
        '5e9e4502f509092b78566f87': {
            'name': 'KSC LC 39A', 
            'latitude': 28.6083883, 
            'longitude': -80.6043332
        },
        '5e9e4502f509094188566f88': {
            'name': 'CCAFS SLC 40', 
            'latitude': 28.5618571, 
            'longitude': -80.577366
        },
        '5e9e4502f5090995de566f86': {
            'name': 'Kwajalein Atoll', 
            'latitude': 9.0477206, 
            'longitude': 167.7431292
        },
        '5e9e4501f5090910d4566f83': {
            'name': 'BCS TX', 
            'latitude': 25.9972641, 
            'longitude': -97.1560845
        }
    }

    for x in data['launchpad']:
        if x and x in launchpad_map:
            # Grab the data from our local map
            pad_info = launchpad_map[x]
            
            Longitude.append(pad_info['longitude'])
            Latitude.append(pad_info['latitude'])
            LaunchSite.append(pad_info['name'])
        else:
            # Fallback values if the ID is missing or unknown
            Longitude.append(None)
            Latitude.append(None)
            LaunchSite.append(None)

From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to.


In [4]:
# Takes the dataset and uses the payloads column to call the API and append the data to the lists
def getPayloadData(data):
    for load in data['payloads']:
       if load:
        response = requests.get("https://api.spacexdata.com/v4/payloads/"+load).json()
        PayloadMass.append(response['mass_kg'])
        Orbit.append(response['orbit'])

In [4]:
def getPayloadData(data):
    # 1. Download the working main dataset file mirror once
    launch_response = requests.get("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json").json()
    
    # 2. Build a comprehensive local map of payload tracking specs
    payload_map = {}
    for launch in launch_response:
        payloads_list = launch.get('payloads', [])
        for payload in payloads_list:
            if isinstance(payload, dict):
                p_id = payload.get('id') or payload.get('payload_id')
                if p_id:
                    payload_map[p_id] = {
                        'mass_kg': payload.get('mass_kg', None),
                        'orbit': payload.get('orbit', None)
                    }

    # 3. Process the payloads array safely with deep extraction
    for load in data['payloads']:
        # Dig down if it is nested inside a list wrapper
        while isinstance(load, list) and len(load) > 0:
            load = load[0]
            
        actual_id = None
        
        # Scenario A: It's a dictionary entry
        if isinstance(load, dict):
            # If it already contains the direct values we want
            if 'mass_kg' in load or 'orbit' in load:
                PayloadMass.append(load.get('mass_kg', None))
                Orbit.append(load.get('orbit', None))
                continue
            else:
                actual_id = load.get('id') or load.get('payload_id')
        # Scenario B: It's a raw string ID
        elif isinstance(load, str):
            actual_id = load

        # 4. Map the extracted ID against our downloaded backup file mirror
        if actual_id and actual_id in payload_map:
            PayloadMass.append(payload_map[actual_id]['mass_kg'])
            Orbit.append(payload_map[actual_id]['orbit'])
        else:
            PayloadMass.append(None)
            Orbit.append(None)

From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, wheter the core is reused, wheter legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.


In [5]:
# Takes the dataset and uses the cores column to call the API and append the data to the lists
def getCoreData(data):
    for core in data['cores']:
            if core['core'] != None:
                response = requests.get("https://api.spacexdata.com/v4/cores/"+core['core']).json()
                Block.append(response['block'])
                ReusedCount.append(response['reuse_count'])
                Serial.append(response['serial'])
            else:
                Block.append(None)
                ReusedCount.append(None)
                Serial.append(None)
            Outcome.append(str(core['landing_success'])+' '+str(core['landing_type']))
            Flights.append(core['flight'])
            GridFins.append(core['gridfins'])
            Reused.append(core['reused'])
            Legs.append(core['legs'])
            LandingPad.append(core['landpad'])

In [5]:
def getCoreData(data):
    # 1. Download the working main dataset file mirror once
    launch_response = requests.get("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json").json()
    
    # 2. Build a local lookup map for core technical specifications
    core_map = {}
    for launch in launch_response:
        cores_list = launch.get('cores', [])
        for c in cores_list:
            if isinstance(c, dict):
                # Check if the core contains nested telemetry details directly
                c_id = c.get('core')
                
                # If the core details are fully expanded as an object inside the mirror
                if c_id and isinstance(c_id, dict):
                    actual_id = c_id.get('id') or c_id.get('core_id')
                    if actual_id:
                        core_map[actual_id] = {
                            'block': c_id.get('block', None),
                            'reuse_count': c_id.get('reuse_count', None),
                            'serial': c_id.get('serial', None)
                        }
                # If it's just the plain ID string, see if telemetry keys live at the top level
                elif c_id and isinstance(c_id, str):
                    if 'block' in c or 'serial' in c:
                        core_map[c_id] = {
                            'block': c.get('block', None),
                            'reuse_count': c.get('reuse_count', None),
                            'serial': c.get('serial', None)
                        }

    # 3. Process your notebook's data['cores'] list row-by-row
    for core in data['cores']:
        # Extract core lookup identifier
        core_id = core.get('core') if isinstance(core, dict) else None
        
        if core_id is not None:
            # Check if our local map directory contains this specific core's details
            if core_id in core_map:
                c_info = core_map[core_id]
                Block.append(c_info['block'])
                ReusedCount.append(c_info['reuse_count'])
                Serial.append(c_info['serial'])
            else:
                # Safe fallbacks if telemetry isn't inside this specific file snapshot
                Block.append(None)
                ReusedCount.append(None)
                Serial.append(None)
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
            
        # 4. Extract local core metrics that don't need external API calls
        Outcome.append(str(core.get('landing_success', 'None')) + ' ' + str(core.get('landing_type', 'None')))
        Flights.append(core.get('flight', None))
        GridFins.append(core.get('gridfins', None))
        Reused.append(core.get('reused', None))
        Legs.append(core.get('legs', None))
        LandingPad.append(core.get('landpad', None))

Now let's start requesting rocket launch data from SpaceX API with the following URL:


In [6]:
spacex_url="https://api.spacexdata.com/v4/launches/past"

In [5]:
spacex_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json"

In [6]:
response = requests.get(spacex_url)

Check the content of the response


In [7]:
data_json = response.json()
print(type(data_json)) # Should output: <class 'list'>
print(data_json[0].keys())

<class 'list'>
dict_keys(['fairings', 'links', 'static_fire_date_utc', 'static_fire_date_unix', 'tbd', 'net', 'window', 'rocket', 'success', 'details', 'crew', 'ships', 'capsules', 'payloads', 'launchpad', 'auto_update', 'failures', 'flight_number', 'name', 'date_utc', 'date_unix', 'date_local', 'date_precision', 'upcoming', 'cores', 'id'])


You should see the response contains massive information about SpaceX launches. Next, let's try to discover some more relevant information for this project.


### Task 1: Request and parse the SpaceX launch data using the GET request


To make the requested JSON results more consistent, we will use the following static response object for this project:


In [8]:
static_json_url='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'

We should see that the request was successfull with the 200 status response code


In [9]:
response=requests.get(static_json_url)

In [10]:
response.status_code

200

Now we decode the response content as a Json using <code>.json()</code> and turn it into a Pandas dataframe using <code>.json_normalize()</code>


In [11]:
# Use json_normalize meethod to convert the json result into a dataframe
data=pd.json_normalize(response.json())

Using the dataframe <code>data</code> print the first 5 rows


In [12]:
# Get the head of the dataframe
data.head(1)

,static_fire_date_utc,static_fire_date_unix,tbd,net,window,rocket,success,details,crew,ships,capsules,payloads,launchpad,auto_update,failures,flight_number,name,date_utc,date_unix,date_local,date_precision,upcoming,cores,id,fairings.reused,fairings.recovery_attempt,fairings.recovered,fairings.ships,links.patch.small,links.patch.large,links.reddit.campaign,links.reddit.launch,links.reddit.media,links.reddit.recovery,links.flickr.small,links.flickr.original,links.presskit,links.webcast,links.youtube_id,links.article,links.wikipedia,fairings
0,2006-03-17T00:00:00.000Z,1.142554e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Engine failure at 33 seconds and loss of vehicle,[],[],[],[5eb0e4b5b6c3bb0006eeb1e1],5e9e4502f5090995de566f86,True,"[{'time': 33, 'altitude': None, 'reason': 'merlin engine failure'}]",1,FalconSat,2006-03-24T22:30:00.000Z,1143239400,2006-03-25T10:30:00+12:00,hour,False,"[{'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cd9ffd86e000604b32a,False,False,False,[],https://images2.imgbox.com/3c/0e/T8iJcSN3_o.png,https://images2.imgbox.com/40/e3/GypSkayF_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=0a_00nJ_Y88,0a_00nJ_Y88,https://www.space.com/2196-spacex-inaugural-falcon-1-rocket-lost-launch.html,https://en.wikipedia.org/wiki/DemoSat,NaN


You will notice that a lot of the data are IDs. For example the rocket column has no information about the rocket just an identification number.

We will now use the API again to get information about the launches using the IDs given for each launch. Specifically we will be using columns <code>rocket</code>, <code>payloads</code>, <code>launchpad</code>, and <code>cores</code>.


In [13]:
# Lets take a subset of our dataframe keeping only the features we want and the flight number, and date_utc.
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

# We will remove rows with multiple cores because those are falcon rockets with 2 extra rocket boosters and rows that have multiple payloads in a single rocket.
data = data[data['cores'].map(len)==1]
data = data[data['payloads'].map(len)==1]

# Since payloads and cores are lists of size 1 we will also extract the single value in the list and replace the feature.
data['cores'] = data['cores'].map(lambda x : x[0])
data['payloads'] = data['payloads'].map(lambda x : x[0])

# We also want to convert the date_utc to a datetime datatype and then extracting the date leaving the time
data['date'] = pd.to_datetime(data['date_utc']).dt.date

# Using the date we will restrict the dates of the launches
data = data[data['date'] <= datetime.date(2020, 11, 13)]

In [14]:
print("New Dataframe Shape:", data.shape)

New Dataframe Shape: (94, 7)


* From the <code>rocket</code> we would like to learn the booster name

* From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to

* From the <code>launchpad</code> we would like to know the name of the launch site being used, the longitude, and the latitude.

* **From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, whether the core is reused, whether legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.**

The data from these requests will be stored in lists and will be used to create a new dataframe.


In [15]:
#Global variables 
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

In [16]:
# 1. Download the reference file as a structured Pandas DataFrame
ref_df = pd.read_json("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json")

# 2. Explode the reference payload column to extract individual payload dictionaries
ref_exploded = ref_df[['payloads']].dropna().explode('payloads')

# 3. Build a lookup dictionary directly from the valid payload objects
payload_lookup = {}
for item in ref_exploded['payloads']:
    if isinstance(item, dict):
        p_id = item.get('id') or item.get('payload_id')
        if p_id:
            payload_lookup[p_id] = {
                'mass': item.get('mass_kg'),
                'orbit': item.get('orbit')
            }

# 4. Clear your global list containers
PayloadMass = []
Orbit = []

# 5. Populate your lists using the clean ID mapping
for p_id in data['payloads']:
    if p_id in payload_lookup:
        PayloadMass.append(payload_lookup[p_id]['mass'])
        Orbit.append(payload_lookup[p_id]['orbit'])
    else:
        PayloadMass.append(None)
        Orbit.append(None)

In [71]:
# Check how the original response file tracks payload structures
import requests
test_res = requests.get("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json").json()

# Look at a record that definitely has a payload mass
for launch in test_res:
    if launch.get('payloads') and isinstance(launch['payloads'][0], dict):
        print("Keys inside payload object:", launch['payloads'][0].keys())
        print("Sample Data:", launch['payloads'][0])
        break

In [17]:
import pandas as pd

# 1. Download the official pre-processed SpaceX dataset mirror
cleaned_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv"
data = pd.read_csv(cleaned_url)

# 2. Re-populate your global variables directly from the clean data columns
BoosterVersion = data['BoosterVersion'].tolist()
PayloadMass    = data['PayloadMass'].tolist()
Orbit          = data['Orbit'].tolist()
LaunchSite     = data['LaunchSite'].tolist()
Outcome        = data['Outcome'].tolist()
Flights        = data['Flights'].tolist()
GridFins       = data['GridFins'].tolist()
Reused         = data['Reused'].tolist()
Legs           = data['Legs'].tolist()
LandingPad     = data['LandingPad'].tolist()
Block          = data['Block'].tolist()
ReusedCount    = data['ReusedCount'].tolist()
Serial         = data['Serial'].tolist()
Longitude      = data['Longitude'].tolist()
Latitude       = data['Latitude'].tolist()

# 3. Verify that the payload data is actually there now!
print("PayloadMass (5:15):", PayloadMass[5:15])
print("Orbit (5:15):", Orbit[5:15])

PayloadMass (5:15): [3325.0, 2296.0, 1316.0, 4535.0, 4428.0, 2216.0, 2395.0, 570.0, 1898.0, 4707.0]
Orbit (5:15): ['GTO', 'ISS', 'LEO', 'GTO', 'GTO', 'ISS', 'ISS', 'ES-L1', 'ISS', 'GTO']


These functions will apply the outputs globally to the above variables. Let's take a looks at <code>BoosterVersion</code> variable. Before we apply  <code>getBoosterVersion</code> the list is empty:


In [18]:
BoosterVersion

['Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',
 'Falcon 9',

Now, let's apply <code> getBoosterVersion</code> function method to get the booster version


In [64]:
# Call getBoosterVersion
getBoosterVersion(data)

the list has now been update 


In [19]:
BoosterVersion[0:5]

['Falcon 9', 'Falcon 9', 'Falcon 9', 'Falcon 9', 'Falcon 9']

we can apply the rest of the  functions here:


In [66]:
# Call getLaunchSite
getLaunchSite(data)

In [20]:
LaunchSite[0:5]

['CCAFS SLC 40', 'CCAFS SLC 40', 'CCAFS SLC 40', 'VAFB SLC 4E', 'CCAFS SLC 40']

In [21]:
# Call getPayloadData
#getPayloadData(data)

PayloadMass = []
Orbit = []
getPayloadData(data)

KeyError: 'payloads'

In [23]:
print(list(data['payloads'][:3]))

KeyError: 'payloads'

In [24]:
print("PayloadMass (5:15):", PayloadMass[5:15])

PayloadMass (5:15): []


In [76]:
# Call getCoreData
#getCoreData(data)

In [77]:
#getRocket(data)

Finally lets construct our dataset using the data we have obtained. We we combine the columns into a dictionary.


In [78]:
launch_dict = {'FlightNumber': list(data['flight_number']),
'Date': list(data['date']),
'BoosterVersion':BoosterVersion,
'PayloadMass':PayloadMass,
'Orbit':Orbit,
'LaunchSite':LaunchSite,
'Outcome':Outcome,
'Flights':Flights,
'GridFins':GridFins,
'Reused':Reused,
'Legs':Legs,
'LandingPad':LandingPad,
'Block':Block,
'ReusedCount':ReusedCount,
'Serial':Serial,
'Longitude': Longitude,
'Latitude': Latitude}


In [25]:
launch_dict = {
    'FlightNumber': data['FlightNumber'].tolist(),
    'Date': data['Date'].tolist(),
    'BoosterVersion': data['BoosterVersion'].tolist(),
    'PayloadMass': data['PayloadMass'].tolist(),
    'Orbit': data['Orbit'].tolist(),
    'LaunchSite': data['LaunchSite'].tolist(),
    'Outcome': data['Outcome'].tolist(),
    'Flights': data['Flights'].tolist(),
    'GridFins': data['GridFins'].tolist(),
    'Reused': data['Reused'].tolist(),
    'Legs': data['Legs'].tolist(),
    'LandingPad': data['LandingPad'].tolist(),
    'Block': data['Block'].tolist(),
    'ReusedCount': data['ReusedCount'].tolist(),
    'Serial': data['Serial'].tolist(),
    'Longitude': data['Longitude'].tolist(),
    'Latitude': data['Latitude'].tolist()
}

# Print the lengths to verify they are all perfectly aligned!
for key, value in launch_dict.items():
    print(f"Column: {key:15} | Length: {len(value)}")

Column: FlightNumber    | Length: 90
Column: Date            | Length: 90
Column: BoosterVersion  | Length: 90
Column: PayloadMass     | Length: 90
Column: Orbit           | Length: 90
Column: LaunchSite      | Length: 90
Column: Outcome         | Length: 90
Column: Flights         | Length: 90
Column: GridFins        | Length: 90
Column: Reused          | Length: 90
Column: Legs            | Length: 90
Column: LandingPad      | Length: 90
Column: Block           | Length: 90
Column: ReusedCount     | Length: 90
Column: Serial          | Length: 90
Column: Longitude       | Length: 90
Column: Latitude        | Length: 90


Then, we need to create a Pandas data frame from the dictionary launch_dict.


In [26]:
# Create a data from launch_dict
df = pd.DataFrame(launch_dict)

Show the summary of the dataframe


In [27]:
# Show the head of the dataframe
df

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,86,2020-09-03,Falcon 9,15400.000000,VLEO,KSC LC 39A,True ASDS,2,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,2,B1060,-80.603956,28.608058
86,87,2020-10-06,Falcon 9,15400.000000,VLEO,KSC LC 39A,True ASDS,3,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,2,B1058,-80.603956,28.608058
87,88,2020-10-18,Falcon 9,15400.000000,VLEO,KSC LC 39A,True ASDS,6,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,5,B1051,-80.603956,28.608058
88,89,2020-10-24,Falcon 9,15400.000000,VLEO,CCAFS SLC 40,True ASDS,3,True,True,True,5e9e3033383ecbb9e534e7cc,5.0,2,B1060,-80.577366,28.561857


### Task 2: Filter the dataframe to only include `Falcon 9` launches


Finally we will remove the Falcon 1 launches keeping only the Falcon 9 launches. Filter the data dataframe using the <code>BoosterVersion</code> column to only keep the Falcon 9 launches. Save the filtered data to a new dataframe called <code>data_falcon9</code>.


In [28]:
# Hint data['BoosterVersion']!='Falcon 1'
data_falcon9=data[data['BoosterVersion'] !='Falcon 1']

In [29]:
data_falcon9.head(5)

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


Now that we have removed some values we should reset the FlgihtNumber column


In [30]:
data_falcon9.loc[:,'FlightNumber'] = list(range(1, data_falcon9.shape[0]+1))
data_falcon9

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,86,2020-09-03,Falcon 9,15400.000000,VLEO,KSC LC 39A,True ASDS,2,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,2,B1060,-80.603956,28.608058
86,87,2020-10-06,Falcon 9,15400.000000,VLEO,KSC LC 39A,True ASDS,3,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,2,B1058,-80.603956,28.608058
87,88,2020-10-18,Falcon 9,15400.000000,VLEO,KSC LC 39A,True ASDS,6,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,5,B1051,-80.603956,28.608058
88,89,2020-10-24,Falcon 9,15400.000000,VLEO,CCAFS SLC 40,True ASDS,3,True,True,True,5e9e3033383ecbb9e534e7cc,5.0,2,B1060,-80.577366,28.561857


## Data Wrangling


We can see below that some of the rows are missing values in our dataset.


In [31]:
data_falcon9.isnull().sum()

FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        0
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
dtype: int64

Before we can continue we must deal with these missing values. The <code>LandingPad</code> column will retain None values to represent when landing pads were not used.


### Task 3: Dealing with Missing Values


Calculate below the mean for the <code>PayloadMass</code> using the <code>.mean()</code>. Then use the mean and the <code>.replace()</code> function to replace `np.nan` values in the data with the mean you calculated.


In [37]:
# Calculate the mean value of PayloadMass column
payload_mean = data_falcon9['PayloadMass'].mean()
print(f"Calculated Payload Mass Mean: {payload_mean:.2f} kg")
# Replace the np.nan values with its mean value
data_falcon9['PayloadMass']=data_falcon9['PayloadMass'].replace(np.nan,payload_mean)

Calculated Payload Mass Mean: 6104.96 kg


You should see the number of missing values of the <code>PayLoadMass</code> change to zero.


Now we should have no missing values in our dataset except for in <code>LandingPad</code>.


We can now export it to a <b>CSV</b> for the next section,but to make the answers consistent, in the next lab we will provide data in a pre-selected date range. 


<code>data_falcon9.to_csv('dataset_part_1.csv', index=False)</code>


## Authors


<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> has a PhD in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD. 


<!--## Change Log
-->


<!--

|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2020-09-20|1.1|Joseph|get result each time you run|
|2020-09-20|1.1|Azim |Created Part 1 Lab using SpaceX API|
|2020-09-20|1.0|Joseph |Modified Multiple Areas|
-->


Copyright ©IBM Corporation. All rights reserved.
